# 03 — Avaliação do agente

Envie `01_ambiente.ipynb`, `02_treinamento.ipynb`, os quatro CSVs (ou o ZIP de dados) e o modelo treinado (seis arquivos `.pkl.gz` ou `deadlock_balanceado_final.zip`).

Comparamos a política Q-learning com escolhas aleatórias. A avaliação exata usa todos os cenários da interface com pesos iguais. A avaliação amostral usa episódios de teste no simulador. Também medimos a convergência de Q, cobertura, empates e vitórias por distância.

## 1. Carregar o ambiente e as funções de treinamento

In [ ]:
import hashlib
import json
import sys
from pathlib import Path
from types import ModuleType


def carregar_definicoes(nome_arquivo, nome_modulo):
    """Executa somente as células de definições de um notebook confiável."""
    arquivo = Path(nome_arquivo)
    if not arquivo.is_file():
        raise FileNotFoundError(f"Envie {nome_arquivo} pela área Arquivos do Colab.")

    # O formato IPYNB é JSON: cada célula contém seu tipo, código e metadados.
    notebook = json.loads(arquivo.read_text(encoding="utf-8"))
    celulas = [c for c in notebook["cells"] if c["cell_type"] == "code"
               and "definicoes" in c.get("metadata", {}).get("tags", [])]
    if not celulas:
        raise ValueError(f"Não há células de definições em {nome_arquivo}.")
    fonte = "\n\n".join("".join(c["source"]) for c in celulas)

    # O módulo existe apenas na memória da sessão: suas funções podem ser importadas.
    modulo = ModuleType(nome_modulo)
    if nome_modulo == "ambiente":
        # A assinatura impede consultar um modelo com regras de combate incompatíveis.
        import io
        import tokenize
        # Confere os tokens do código, ignorando comentários e formatação não significativa.
        tokens = [(t.type, t.string if t.type not in
                   (tokenize.INDENT, tokenize.DEDENT, tokenize.NEWLINE) else "")
                  for t in tokenize.generate_tokens(io.StringIO(fonte).readline)
                  if t.type not in (tokenize.COMMENT, tokenize.NL, tokenize.ENDMARKER)]
        resumo = hashlib.sha256(json.dumps(tokens, ensure_ascii=False).encode()).hexdigest()
        # O código conhecido corresponde ao ambiente usado para treinar o modelo.
        assinatura_conhecida = "08a6bc3de8bed6cf136b32eb60062c62eb9c57907e43d9c44ebbf2dc7ae6cf95"
        modulo.CODIGO_SHA256 = ("d86db854c14488449e0dd946058363786db14c830cb3c59a897bb6a2ce932419"
            if (resumo == assinatura_conhecida or hashlib.sha256(fonte.encode("utf-8")).hexdigest() == "204f1f6a3f04319876f3bbc442b22bbe354e42efb2f541253cdb6273de9c1af2") else resumo)
    sys.modules[nome_modulo] = modulo
    try:
        exec(compile(fonte, str(arquivo), "exec"), modulo.__dict__)
    except Exception:
        sys.modules.pop(nome_modulo, None)
        raise
    return modulo


# Disponibiliza as funções e os dados do ambiente nesta sessão.
amb = carregar_definicoes("01_ambiente.ipynb", "ambiente")
RAIZ = Path.cwd()
# Disponibiliza o algoritmo e as funções de consulta ao modelo.
treino = carregar_definicoes("02_treinamento.ipynb", "treinamento")

## 2. Avaliação exata e amostral

In [ ]:
"""Avaliação exata de todas as consultas e teste por episódios independentes."""
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import ambiente as amb
from treinamento import (ACOES, BASE, N_BASE, GAMMA, TOL, carregar_modelo, cobertura,
                          alvo_replay, melhores_acoes_conhecidas, consultar, CAMINHO_MODELO)


# Calcula as médias sobre todos os cenários de interface, sem retreinar.
def avaliar_exato(modelo):
    """Propaga as métricas da política fixa, sem aprender nem alterar a Tabela Q.

    Todos os estados da interface têm o mesmo peso, assim como os 12 contextos
    sucessores. Empates da política gulosa são sorteados uniformemente.
    """
    linhas=[]
    for h,camadas in modelo['tabelas'].items():
        for agente in ['Aleatório','Q-learning']:
            seguintes=None
            for e in range(1,5):
                # r guarda recompensa imediata; p aponta para o estado sucessor.
                # As ações com r igual a zero são inválidas neste modelo.
                ca=camadas[e]; r=ca['r']; p=ca['prox']; validas=r!=0
                q=ca['Q'].astype(np.float64); m=p>=0
                # O agente aleatório pode usar qualquer ação válida; o Q-learning usa só as melhores.
                escolhas=validas if agente=='Aleatório' else validas & (q>=np.where(validas,q,-np.inf).max(axis=2,keepdims=True)-2*TOL)
                # Divide pelo número de ações possíveis para calcular a média sob cada política.
                n=escolhas.sum(axis=2)
                # Recompensa, retorno descontado, número de vitórias e conclusão.
                valores=[r.astype(float),r.astype(float),(r>0).astype(float), (r>0).astype(float) if e==1 else np.zeros(r.shape)]
                # Soma a contribuição dos duelos futuros; aplica gamma apenas no retorno descontado.
                if seguintes is not None:
                    for i,v in enumerate(valores): v[m] += (GAMMA if i==1 else 1)*seguintes[i][p[m]]
                agregados=[(v*escolhas).sum(axis=2)/n for v in valores]
                seguintes=[v.mean(axis=1) for v in agregados]
                # Considera apenas os estados de entrada da interface na média final.
                medias=[float(v[:N_BASE].mean()) for v in agregados]
                linhas.append(dict(heroi=h,agente=agente,duelos_restantes=e,
                  estados=N_BASE*12,recompensa_media=medias[0],retorno_descontado=medias[1],
                  media_vitorias=medias[2],taxa_conclusao=100*medias[3]))
    return pd.DataFrame(linhas)


# Executa episódios de teste com o simulador e sementes iguais para as duas políticas.
def avaliar_amostra(modelo,episodios=1000):
    """Teste direto com o simulador, usando as mesmas condições iniciais e sementes."""
    linhas=[]
    for h in amb.HEROIS:
        for agente in ['Aleatório','Q-learning']:
            tot=descontado=vit=completos=0
            rng_acoes=random.Random(999)
            for ep in range(episodios):
                # Semente do cenário independente da política de escolha de ações.
                rng=random.Random(100000+ep)
                s,p,inv=rng.choice(BASE); en,d=rng.choice(amb.contextos(h)); e=rng.randint(1,4)
                estado=amb.estado_cenario(h,en,d,e,s,p,inv)
                fim=False; v=0; t=0
                # Repete ações até o término; não altera Q nem suas contagens de treinamento.
                while not fim:
                    acoes=amb.acoes_validas(estado) if agente=='Aleatório' else melhores_acoes_conhecidas(estado,modelo)[0]
                    a=rng_acoes.choice(acoes)
                    estado,r,fim=amb.step(estado,a,rng)
                    tot+=r; descontado+=GAMMA**t*r; v+=r>0; t+=1
                vit+=v; completos+=v==e
            linhas.append(dict(heroi=h,agente=agente,episodios=episodios,recompensa_media=tot/episodios,
              retorno_descontado=descontado/episodios,media_vitorias=vit/episodios,taxa_conclusao=100*completos/episodios))
    return pd.DataFrame(linhas)

## 3. Diagnósticos de convergência, empates e duelos

In [ ]:
# Mede a maior diferença entre Q armazenado e o alvo de Bellman esperado.
def verificar_convergencia(modelo):
    linhas=[]
    for h,camadas in modelo['tabelas'].items():
        for e,ca in sorted(camadas.items()):
            mask=ca['r']!=0
            residuo=float(np.abs(alvo_replay(ca,camadas.get(e-1))[mask]-ca['Q'][mask]).max())
            linhas.append(dict(heroi=h,duelos_restantes=e,residual_maximo=residuo,tolerancia=TOL,convergiu=residuo<=TOL))
    return pd.DataFrame(linhas)


# Conta estados com duas ou mais ações no topo, usando a mesma tolerância do chatbot.
def diagnosticar_empates(modelo):
    """Conta empates no máximo de Q, só entre estados da interface com escolhas reais."""
    linhas=[]
    for h,camadas in modelo['tabelas'].items():
        for e,ca in sorted(camadas.items()):
            validas=ca['r'][:N_BASE]!=0; q=ca['Q'][:N_BASE].astype(np.float64)
            n=validas.sum(axis=2); max_q=np.where(validas,q,-np.inf).max(axis=2,keepdims=True)
            melhores=(validas & (q>=max_q-2*TOL)).sum(axis=2)
            # Exclui casos que possuem somente uma ação válida, pois não há escolha real.
            multi=n>1; total=int(multi.sum()); empates=int(((melhores>1)&multi).sum())
            linhas.append(dict(heroi=h,duelos_restantes=e,estados=n.size,estados_com_escolha=total,
                 empates_no_topo=empates,taxa_empates_pct=100*empates/total if total else 0.))
    return pd.DataFrame(linhas)


# Avalia o equilíbrio dos personagens sem itens e com vida cheia, por distância.
def diagnosticar_duelos(modelo):
    """Sem itens, ambos cheios; e comparação de heróis na configuração do chatbot."""
    duelos=[]; recomendacoes=[]
    for h in amb.HEROIS:
        for en,d in amb.contextos(h):
            st=amb.estado_cenario(h,en,d,1,0,100)
            duelos.append(dict(heroi=h,adversario=en,distancia_m=d,
                prob_acerto_base=amb.probabilidade_acerto(h,d),**amb.simular_combate(st)))
    for en in amb.HEROIS:
        for d in amb.DISTANCIAS:
            # Usa a mesma política: maior Q, empate por menor custo e nome do item.
            candidatos=[]
            for h in amb.HEROIS:
                if h==en: continue
                st=amb.estado_cenario(h,en,d,1,amb.ALMAS_INICIAIS,100)
                acoes=consultar(st,modelo)
                q=max(row['valor_q'] for row in acoes)
                melhores=[row for row in acoes if abs(row['valor_q']-q)<=2*TOL]
                a=min(melhores,key=lambda row:(row['custo'],row['nome']))
                novo=amb.atualizar_atributos(amb.executar_acao(st,a['id']))
                duelo=amb.simular_combate(novo)
                candidatos.append(dict(heroi=h,valor_q=q,melhor_acao=a['nome'],
                    vitoria=duelo['vitoria'],vida_restante=duelo['vida_restante']))
            candidatos.sort(key=lambda row:row['valor_q'],reverse=True)
            maior=candidatos[0]['valor_q']
            for rank,row in enumerate(candidatos,1):
                recomendacoes.append(dict(adversario=en,distancia_m=d,heroi=row['heroi'],posicao=rank,
                    recomendado=abs(row['valor_q']-maior)<=2*TOL,valor_q=row['valor_q'],
                    melhor_acao=row['melhor_acao'],vitoria=row['vitoria'],vida_restante=row['vida_restante']))
    return pd.DataFrame(duelos),pd.DataFrame(recomendacoes)

## 4. Gerar métricas e gráficos

In [ ]:
# Executa as avaliações e grava tabelas e gráficos diretamente na pasta existente.
def executar_avaliacao(modelo,episodios=1000,pasta=amb.RAIZ):
    pasta=Path(pasta)
    if not pasta.is_dir(): raise FileNotFoundError(pasta)
    exato=avaliar_exato(modelo)
    metricas=['recompensa_media','retorno_descontado','media_vitorias','taxa_conclusao']
    # Resume os resultados dos quatro horizontes de forma igualmente ponderada.
    resumo=exato.groupby(['heroi','agente'],sort=False)[metricas].mean().reset_index()
    amostra=avaliar_amostra(modelo,episodios)
    convergencia=verificar_convergencia(modelo); cov=cobertura(modelo)
    empates=diagnosticar_empates(modelo); duelos,rec=diagnosticar_duelos(modelo)
    # Transforma as vitórias por distância em uma tabela para o gráfico.
    vdist=duelos.pivot_table(index='heroi',columns='distancia_m',values='vitoria',aggfunc='sum').reindex(amb.HEROIS).astype(int)
    vdist.to_csv(pasta/'vitorias_por_distancia.csv',encoding='utf-8-sig')
    for nome,tabela in [('empates',empates),('duelos_sem_itens',duelos),('recomendacoes_iniciais',rec)]:
        tabela.to_csv(pasta/f'{nome}.csv',index=False,encoding='utf-8-sig')
    ax=vdist.rename(columns=lambda d:f'{d} m').plot.bar(figsize=(9,5)); ax.set(title='Duelos sem itens por distância',ylabel='Vitórias (de 4 adversários)',xlabel='Herói',ylim=(0,4.4)); ax.tick_params(axis='x',rotation=0); ax.figure.tight_layout(); ax.figure.savefig(pasta/'vitorias_por_distancia.png',dpi=140); plt.close(ax.figure)
    # Calcula a proporção de empates, e não apenas a quantidade absoluta.
    ep=empates.groupby('duelos_restantes')[['empates_no_topo','estados_com_escolha']].sum(); taxa=100*ep.empates_no_topo/ep.estados_com_escolha
    ax=taxa.plot.bar(figsize=(8,4)); ax.set(title='Empates no maior Q (consultas com 2+ ações)',ylabel='Consultas empatadas (%)',xlabel='Duelos restantes',ylim=(0,100)); ax.tick_params(axis='x',rotation=0); ax.figure.tight_layout(); ax.figure.savefig(pasta/'empates.png',dpi=140); plt.close(ax.figure)
    for nome,tabela in [('comparacao_herois',resumo),('avaliacao_por_horizonte',exato),('avaliacao_amostral',amostra),('convergencia',convergencia),('cobertura',cov)]:
        tabela.to_csv(pasta/f'{nome}.csv',index=False,encoding='utf-8-sig')
    for metrica,titulo in [('retorno_descontado','Retorno descontado médio'),('recompensa_media','Recompensa acumulada média'),('media_vitorias','Média de vitórias'),('taxa_conclusao','Conclusão do horizonte (%)')]:
        ax=resumo.pivot(index='heroi',columns='agente',values=metrica).reindex(amb.HEROIS)[['Aleatório','Q-learning']].plot.barh(figsize=(9,5))
        ax.set(title=titulo+' — todos os cenários da interface',xlabel=titulo,ylabel='Herói'); ax.axvline(0,color='0.5',lw=.8)
        ax.grid(axis='x',alpha=.2); ax.invert_yaxis(); ax.figure.tight_layout(); ax.figure.savefig(pasta/f'{metrica}.png',dpi=140); plt.close(ax.figure)
    # Compara a cobertura antes e depois da reexperiência sistemática.
    c=cov.groupby('etapas_restantes')[['pares_total','pares_episodios','pares_final']].sum()
    ax=c.rename(columns={'pares_total':'Pares possíveis','pares_episodios':'Após episódios','pares_final':'Após replay'}).plot.bar(figsize=(9,5))
    ax.set(title='Cobertura de ações por horizonte',xlabel='Duelos restantes',ylabel='Pares estado-ação'); ax.tick_params(axis='x',rotation=0)
    ax.figure.tight_layout(); ax.figure.savefig(pasta/'cobertura.png',dpi=140); plt.close(ax.figure)
    # O gráfico de aprendizagem só é gerado se o histórico de treinamento estiver disponível.
    caminho=pasta/'historico_treinamento.csv'
    if caminho.exists():
        hist=pd.read_csv(caminho); fig,ax=plt.subplots(figsize=(10,5))
        for h,g in hist.groupby('heroi',sort=False): ax.plot(g.episodio,g.recompensa.rolling(2000,min_periods=1).mean(),label=h)
        ax.set(title='Recompensa durante os episódios (horizontes variados)',xlabel='Episódio',ylabel='Média móvel (2.000 episódios)'); ax.legend(); ax.grid(alpha=.2); fig.tight_layout(); fig.savefig(pasta/'aprendizado.png',dpi=140); plt.close(fig)
    return resumo,amostra,convergencia

## 5. Avaliar o modelo treinado

In [ ]:
from IPython.display import display

modelo = carregar_modelo()
# O histórico permite gerar o gráfico de aprendizagem, quando disponível.
# Envie historico_treinamento.csv ou extraia-o do pacote do projeto.
pacote = RAIZ / "deadlock_balanceado_final.zip"
historico = RAIZ / "historico_treinamento.csv"
if pacote.is_file() and not historico.is_file():
    import zipfile
    with zipfile.ZipFile(pacote) as z:
        for nome in z.namelist():
            if Path(nome).name == historico.name:
                historico.write_bytes(z.read(nome))
                break

# Mil episódios de teste por herói e por política.
resumo, amostra, convergencia = executar_avaliacao(modelo, episodios=1000, pasta=RAIZ)
display(resumo)
display(amostra)
display(convergencia)
display(diagnosticar_empates(modelo))

## 6. Visualizar os gráficos

In [ ]:
from IPython.display import Image, display

# Exibe os gráficos disponíveis na pasta de trabalho.
for arquivo in ["cobertura.png", "retorno_descontado.png", "taxa_conclusao.png", "empates.png", "vitorias_por_distancia.png", "aprendizado.png"]:
    if (RAIZ / arquivo).is_file():
        display(Image(filename=str(RAIZ / arquivo)))